# Viewing measurement data


## Setup and helper functions

In [ ]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator

plt.rcParams.update({
    "figure.figsize": (10, 7),
    "axes.grid": True,
})

COLUMNS = [
    "Freq_Hz",
    "Time_s",
    "Eps_real",
    "Eps_imag",
    "Temp_K",
    "MTime_s",
    "Phi_deg",
    "Z_real_Ohm",
    "Z_imag_Ohm",
    "TanPhi",
]


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data" / "Daten final").exists():
            return path
    raise FileNotFoundError("Could not find the project root with data/Daten final.")


def add_code_dir_to_path(project_root):
    code_dir = project_root / "Code"
    code_dir_text = str(code_dir)
    if code_dir_text not in sys.path:
        sys.path.insert(0, code_dir_text)




PROJECT_ROOT = find_project_root()
add_code_dir_to_path(PROJECT_ROOT)
import switch_points
importlib.reload(switch_points)
manual_switch_points = switch_points.MANUAL_SWITCH_POINTS

def load_measurements(data_dir=None, selected_file=None, material_filter="PEI5mgmL"):
    project_root = find_project_root()
    data_dir = Path(data_dir) if data_dir is not None else project_root / "data" / "Daten final"

    if selected_file is None:
        file_paths = sorted(data_dir.glob("*.TXT")) + sorted(data_dir.glob("*.txt"))
    else:
        file_path = data_dir / selected_file
        if not file_path.exists():
            raise FileNotFoundError(f"File not found: {file_path}")
        file_paths = [file_path]

    if selected_file is None and material_filter is not None:
        file_paths = [
            path for path in file_paths
            if path.stem.split("_")[0] == material_filter
        ]

    if not file_paths:
        raise FileNotFoundError(f"No .TXT files found in: {data_dir}")

    frames = []
    for path in file_paths:
        df = pd.read_csv(
            path,
            sep=r"\s+",
            skiprows=4,
            names=COLUMNS,
            encoding="latin1",
            engine="python",
        )

        parts = path.stem.split("_")
        material = parts[0] if len(parts) >= 1 else path.stem
        temperature = parts[1] if len(parts) >= 2 else None
        mode = parts[2] if len(parts) >= 3 else None

        first_freq = df["Freq_Hz"].iloc[0]
        df["Spectrum_ID"] = (df["Freq_Hz"] == first_freq).cumsum() - 1
        df["Spectrum_Number"] = df["Spectrum_ID"] + 1
        df["Point_Number"] = range(1, len(df) + 1)
        df["Material"] = material
        df["Temperature"] = temperature
        df["Mode"] = mode
        df["Source_File"] = path.name

        frames.append(df)

    return pd.concat(frames, ignore_index=True)


def filter_dataset(df, material, temperature, mode):
    return df[
        (df["Material"] == material)
        & (df["Temperature"] == temperature)
        & (df["Mode"] == mode)
    ].copy()


def print_available_datasets(df):
    available = (
        df[["Material", "Temperature", "Mode"]]
        .drop_duplicates()
        .sort_values(["Material", "Temperature", "Mode"])
    )
    print(available.to_string(index=False))

## Load raw data

In [ ]:
df_all = load_measurements()

print(f"Rows: {len(df_all):,}")
print(f"Files: {df_all['Source_File'].nunique()}")
print(df_all.head())

In [ ]:
series_counts = (
    df_all.groupby(["Material", "Temperature", "Mode"], dropna=False)
    .agg(
        Point_Count=("Point_Number", "count"),
        Spectrum_Count=("Spectrum_Number", "nunique"),
        First_MTime_s=("MTime_s", "min"),
        Last_MTime_s=("MTime_s", "max"),
    )
    .reset_index()
    .sort_values(["Material", "Temperature", "Mode"])
)

print(series_counts.to_string(index=False))

## Apply atmosphere switch points


In [ ]:
# Switch points are maintained centrally in Code/switch_points.py.
series_keys = ["Material", "Temperature", "Mode"]

switch_times = []
for dataset_key, dataset in df_all.groupby(series_keys, dropna=False):
    switch_point_number = manual_switch_points.get(dataset_key)
    if switch_point_number is None:
        raise ValueError(f"Missing Switch_Point_Number for series: {dataset_key}")

    switch_point = dataset[dataset["Point_Number"] == switch_point_number]
    next_point = dataset[dataset["Point_Number"] == switch_point_number + 1]
    if switch_point.empty or next_point.empty:
        raise ValueError(f"Switch-point pair not found: {dataset_key}, {switch_point_number}")

    switch_times.append({
        "Material": dataset_key[0],
        "Temperature": dataset_key[1],
        "Mode": dataset_key[2],
        "Switch_Point_Number": int(switch_point_number),
        "Switch_Time_s": float((switch_point["MTime_s"].iloc[0] + next_point["MTime_s"].iloc[0]) / 2),
    })

switch_times = pd.DataFrame(switch_times)
df_all = df_all.drop(columns=["Switch_Point_Number", "Switch_Time_s", "Time_Relative_s"], errors="ignore")
df_all = df_all.merge(switch_times, on=series_keys, how="left")

if df_all["Switch_Time_s"].isna().any():
    missing = df_all[df_all["Switch_Time_s"].isna()][series_keys].drop_duplicates()
    raise ValueError("At least one dataset is missing a Switch_Point_Number:\n" + missing.to_string(index=False))

df_all["Time_Relative_s"] = df_all["MTime_s"] - df_all["Switch_Time_s"]


## Frequency spectra after setting relative time

In [ ]:
material = "PEI5mgmL"
temperature = "50°C"
mode = "Abs"
y_col = "Eps_real"

df_sel = filter_dataset(df_all, material, temperature, mode)

if df_sel.empty:
    print_available_datasets(df_all)
    raise ValueError(f"No data found for {material}, {temperature}, {mode}.")

fig, ax = plt.subplots(figsize=(10, 7))
group_cols = ["Source_File", "Material", "Temperature", "Mode", "Spectrum_ID"]

for _, spec in df_sel.groupby(group_cols, dropna=False):
    spec = spec.sort_values("Freq_Hz")
    ax.plot(
        spec["Freq_Hz"],
        spec[y_col],
        marker="o",
        markersize=2,
        linewidth=1,
        alpha=0.7,
    )

ax.set_xscale("log")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel(y_col)
ax.set_title(f"{y_col} spectra | {material}-{temperature}-{mode}")
ax.grid(True, which="both")
plt.tight_layout()
plt.show()

## Time series at one frequency

In [ ]:
def build_nearest_frequency_time_series(df, material, mode, temperature, target_frequency, value_col):
    df_sel = filter_dataset(df, material, temperature, mode)

    if df_sel.empty:
        return pd.DataFrame(columns=["Time_Relative_s", value_col])

    rows = []
    for _, spec in df_sel.groupby("Spectrum_ID", dropna=False):
        row = spec.loc[(spec["Freq_Hz"] - target_frequency).abs().idxmin()]
        rows.append({
            "Time_Relative_s": spec["Time_Relative_s"].iloc[0],
            value_col: row[value_col],
        })

    return pd.DataFrame(rows).sort_values("Time_Relative_s")


def plot_temperature_time_series(df, material, mode, target_frequency, value_col):
    fig, ax = plt.subplots(figsize=(9, 6))
    temperatures = sorted(df["Temperature"].dropna().unique())

    for temperature in temperatures:
        time_series = build_nearest_frequency_time_series(
            df,
            material=material,
            mode=mode,
            temperature=temperature,
            target_frequency=target_frequency,
            value_col=value_col,
        )

        if time_series.empty:
            continue

        ax.plot(
            time_series["Time_Relative_s"],
            time_series[value_col],
            marker="o",
            label=temperature,
        )

    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Time relative to atmosphere switch (s)")
    ax.set_ylabel(value_col)
    ax.set_title(f"{value_col} at f = {target_frequency} Hz | {material}-{mode}")
    ax.grid(True)
    ax.legend(title="Temperature")
    plt.tight_layout()
    plt.show()


def plot_mode_comparison(df, material, temperature, target_frequency, value_col, modes=("Abs", "Des")):
    fig, ax = plt.subplots(figsize=(9, 6))

    for mode in modes:
        time_series = build_nearest_frequency_time_series(
            df,
            material=material,
            mode=mode,
            temperature=temperature,
            target_frequency=target_frequency,
            value_col=value_col,
        )

        if time_series.empty:
            continue

        ax.plot(
            time_series["Time_Relative_s"],
            time_series[value_col],
            marker="o",
            label=mode,
        )

    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Time relative to atmosphere switch (s)")
    ax.set_ylabel(value_col)
    ax.set_title(f"Abs vs Des | {material}-{temperature}, f = {target_frequency} Hz")
    ax.grid(True)
    ax.legend(title="Mode")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_temperature_time_series(
    df_all,
    material="PEI5mgmL",
    mode="Abs",
    target_frequency=100,
    value_col="Eps_real",
)

In [ ]:
plot_temperature_time_series(
    df_all,
    material="PEI5mgmL",
    mode="Abs",
    target_frequency=100,
    value_col="Eps_imag",
)

In [ ]:
plot_temperature_time_series(
    df_all,
    material="PEI5mgmL",
    mode="Des",
    target_frequency=100,
    value_col="Eps_real",
)

In [ ]:
plot_temperature_time_series(
    df_all,
    material="PEI5mgmL",
    mode="Des",
    target_frequency=100,
    value_col="Eps_imag",
)

In [ ]:
plot_mode_comparison(
    df_all,
    material="PEI5mgmL",
    temperature="50°C",
    target_frequency=100,
    value_col="Eps_imag",
)